# Business Entity Resolution Pipeline

**Single notebook — run only via Kernel → Restart & Run All (or `nbconvert --execute`)**

Sections §0–§9 implement the full pipeline as specified in `implementation.md`.  
All 10 sections run top-to-bottom in one kernel session; no cell should be re-run in isolation.

---

## §0 Setup & Configuration

Imports all libraries, loads `configs/pipeline.yaml`, pins the random seed, and defines all shared dataclasses used by every downstream section.

In [ ]:
# §0.1 — Standard-library and third-party imports
# No network-accessing imports below (validated by the grep check in README.md)
import os
import sys
import re
import csv
import math
import random
import pickle
import unicodedata
import subprocess
import collections
import pathlib
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import yaml

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import Levenshtein
import jellyfish

print("§0.1 imports OK")


In [ ]:
# §0.2 — Load configuration from pipeline.yaml
# Notebook lives at: src/business_entity_resolution/pipeline.ipynb
# Config lives at:   configs/pipeline.yaml  (two levels up)
_NB_DIR = pathlib.Path(os.path.abspath(""))
_CONFIG_PATH = (_NB_DIR / ".." / ".." / "configs" / "pipeline.yaml").resolve()

with open(_CONFIG_PATH, "r", encoding="utf-8") as _f:
    CONFIG = yaml.safe_load(_f)

RUN_MODE   = CONFIG["run_mode"]    # "train" or "test"
RANDOM_SEED = CONFIG["random_seed"]

assert RUN_MODE in ("train", "test"), f"run_mode must be 'train' or 'test', got {RUN_MODE!r}"

_PATHS = CONFIG["paths"]
DATA_DIR   = (_NB_DIR / _PATHS["train_dir" if RUN_MODE == "train" else "test_dir"]).resolve()
OUTPUT_DIR = (_NB_DIR / _PATHS["output_dir"]).resolve()
MODELS_DIR = (_NB_DIR / _PATHS["models_dir"]).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

PREFIX = "train" if RUN_MODE == "train" else "test"

print(f"§0.2 CONFIG loaded  run_mode={RUN_MODE}  seed={RANDOM_SEED}")
print(f"     data_dir  : {DATA_DIR}")
print(f"     output_dir: {OUTPUT_DIR}")


In [ ]:
# §0.3 — Pin random seed everywhere
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"§0.3 Random seed pinned to {RANDOM_SEED}")


In [ ]:
# §0.4 — Dataclass definitions (§4.3 of implementation.md)
# Defined here once so every section below can reference them without imports.

@dataclass
class NormalizedRecord:
    entity_id:          str
    source:             str          # "S1", "S2", or "S3"
    raw_name:           str
    raw_address:        str
    raw_country:        str
    normalized_name:    str
    normalized_address: str
    normalized_country: str
    name_tokens:        List[str]
    address_tokens:     List[str]
    blocking_keys:      List[str]


@dataclass
class CandidatePair:
    source1_entity_id:   str
    candidate_entity_id: str
    candidate_source:    str         # "S2" or "S3"
    matched_blocking_keys: List[str] = field(default_factory=list)


@dataclass
class ScoredPair(CandidatePair):
    features:          Dict[str, float] = field(default_factory=dict)
    match_probability: float            = 0.0
    predicted_label:   bool             = False


@dataclass
class FinalMatch:
    source1_entity_id: str
    matched_entity_ids: List[str] = field(default_factory=list)


@dataclass
class CandidateSetRow:
    source1_entity_id:    str
    candidate_entity_ids: List[str] = field(default_factory=list)

print("§0.4 Dataclasses defined: NormalizedRecord, CandidatePair, ScoredPair, FinalMatch, CandidateSetRow")


In [ ]:
# §0 SANITY CHECK
assert RUN_MODE in ("train", "test")
assert isinstance(RANDOM_SEED, int)
assert DATA_DIR.exists(), f"DATA_DIR missing: {DATA_DIR}"
assert OUTPUT_DIR.exists()
_r  = NormalizedRecord("S1-1","S1","T","A","US","t","a","us",["t"],["a"],[])
_cp = CandidatePair("S1-1", "S2-2", "S2")
_fm = FinalMatch("S1-1", ["S2-2"])
assert _r.entity_id == "S1-1"
assert _cp.candidate_source == "S2"
assert _fm.matched_entity_ids == ["S2-2"]
print("§0 SANITY CHECK PASSED")


---
## §1 Data Ingestion

Reads all source TSVs (and ground truth when in train mode), validates schemas per §4.1/§4.2, and tags every record with its source prefix.

In [ ]:
# §1.1 — Source file loader
def load_source(path: pathlib.Path, expected_prefix: str) -> pd.DataFrame:
    """Read a source TSV, validate columns and entity_id prefix."""
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    required = {"entity_id", "business_name", "business_address", "country"}
    missing = required - set(df.columns)
    assert not missing, f"{path.name}: missing columns {missing}"
    for col in required:
        df[col] = df[col].str.strip()
    df["source"] = df["entity_id"].str.split("-").str[0]
    wrong = (df["source"] != expected_prefix).sum()
    assert wrong == 0, f"{path.name}: {wrong} rows have prefix != {expected_prefix!r}"
    return df


def load_ground_truth(path: pathlib.Path) -> pd.DataFrame:
    """Read train_ground_truth.tsv with tab separator."""
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    for col in ("source1_entity_id", "matched_entity_ids"):
        assert col in df.columns, f"{path.name}: missing column {col!r}"
        df[col] = df[col].str.strip()
    return df

print("§1.1 load_source(), load_ground_truth() defined")


In [ ]:
# §1.2 — Load all source files
print(f"Loading {PREFIX} data files from: {DATA_DIR}")

df_s1 = load_source(DATA_DIR / f"{PREFIX}_source1.tsv", "S1")
df_s2 = load_source(DATA_DIR / f"{PREFIX}_source2.tsv", "S2")
df_s3 = load_source(DATA_DIR / f"{PREFIX}_source3.tsv", "S3")

print(f"  S1: {len(df_s1):,} records")
print(f"  S2: {len(df_s2):,} records")
print(f"  S3: {len(df_s3):,} records")

gt_path = DATA_DIR / "train_ground_truth.tsv"
if RUN_MODE == "train" and gt_path.exists():
    df_gt = load_ground_truth(gt_path)
    print(f"  GT: {len(df_gt):,} rows")
    # Build labels dict and gt_map
    labels: Dict[Tuple[str, str], int] = {}
    gt_map:  Dict[str, List[str]]      = {}
    _loaded_s1_ids = set(df_s1["entity_id"])
    for _, row in df_gt.iterrows():
        s1id = row["source1_entity_id"]
        if s1id not in _loaded_s1_ids:
            continue  # skip GT rows not in the loaded S1 slice
        ms   = row["matched_entity_ids"]
        mids = [m.strip() for m in ms.split(",") if m.strip()] if ms else []
        gt_map[s1id] = mids
        for mid in mids:
            labels[(s1id, mid)] = 1
    print(f"  Labels: {len(labels):,} positive pairs")
else:
    df_gt = None; labels = None; gt_map = None
    print("  GT: not loaded (test mode)")


In [ ]:
# §1 SANITY CHECK
assert len(df_s1) > 0 and len(df_s2) > 0 and len(df_s3) > 0
assert df_s1["entity_id"].str.startswith("S1-").all()
assert df_s2["entity_id"].str.startswith("S2-").all()
assert df_s3["entity_id"].str.startswith("S3-").all()
assert "country" in df_s1.columns  # open-set, not validated against a fixed list
if RUN_MODE == "train":
    assert df_gt is not None and labels is not None
    _gt_s1_ids = set(df_gt["source1_entity_id"])
    _missing_gt = set(df_s1["entity_id"]) - _gt_s1_ids
    assert len(_missing_gt) == 0, f"GT missing {len(_missing_gt)} S1 entity IDs"
print("§1 SANITY CHECK PASSED")


---
## §2 Normalization

Canonicalizes `business_name` and `business_address`. Dispatches via `COUNTRY_RULES` — a dict with named handlers for known countries and a **generic fallback** for any unseen value (open-set, never raises, never drops).

In [ ]:
# §2.1 — Shared normalization utilities (vectorized via pandas str ops)

# Abbreviation tables
_LEGAL_ABBREVS = [
    (r"\bcorp\.?\b",  "corporation"),
    (r"\binc\.?\b",   "incorporated"),
    (r"\bltd\.?\b",   "limited"),
    (r"\bllc\.?\b",   "limited liability company"),
    (r"\bllp\.?\b",   "limited liability partnership"),
    (r"\bpvt\.?\b",   "private"),
    (r"\bco\.?\b",    "company"),
    (r"\bplc\.?\b",   "public limited company"),
    (r"\bintl\.?\b",  "international"),
    (r"\bsrl\.?\b",   "societe responsabilite limitee"),
    (r"\bsa\.?\b",    "societe anonyme"),
    (r"\bsarl\.?\b",  "societe responsabilite limitee"),
]

_ADDR_ABBREVS = [
    (r"\brd\.?\b",   "road"),
    (r"\bst\.?\b",   "street"),
    (r"\bave?\.?\b", "avenue"),
    (r"\bblvd\.?\b", "boulevard"),
    (r"\bdr\.?\b",   "drive"),
    (r"\bln\.?\b",   "lane"),
    (r"\bct\.?\b",   "court"),
    (r"\bpl\.?\b",   "place"),
    (r"\bpkwy\.?\b", "parkway"),
    (r"\bhwy\.?\b",  "highway"),
    (r"\bapt\.?\b",  "apartment"),
    (r"\bste\.?\b",  "suite"),
]


def _base_normalize_series(s: pd.Series) -> pd.Series:
    """Vectorized: lowercase, collapse whitespace, remove special punctuation."""
    s = s.str.lower().str.strip()
    s = s.str.replace(r'[&@#$%*!?;:\'"()\[\]{}]', " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s


def _apply_abbrevs_series(s: pd.Series, table: list) -> pd.Series:
    for pat, repl in table:
        s = s.str.replace(pat, repl, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def tokenize_series(s: pd.Series) -> pd.Series:
    """Split each string into a list of non-empty tokens."""
    return s.str.split()


print("§2.1 vectorized normalization utilities defined")


In [ ]:
# §2.2 — Country-aware dispatch: per-country name/address normalizers
# COUNTRY_RULES dispatch dict — open-set: 'default' handles every unseen value.

def _norm_name_us(s: pd.Series) -> pd.Series:
    s = _base_normalize_series(s)
    s = s.str.replace(r" & ", " and ", regex=False)
    return _apply_abbrevs_series(s, _LEGAL_ABBREVS)

def _norm_addr_us(s: pd.Series) -> pd.Series:
    s = _base_normalize_series(s)
    return _apply_abbrevs_series(s, _ADDR_ABBREVS)

def _norm_name_india(s: pd.Series) -> pd.Series:
    # Devanagari kept intact; base normalization + legal abbrevs only
    s = _base_normalize_series(s)
    return _apply_abbrevs_series(s, _LEGAL_ABBREVS)

def _norm_addr_india(s: pd.Series) -> pd.Series:
    s = _base_normalize_series(s)
    # Drop trailing landmark references
    s = s.str.replace(r"\bnear\b.*$", "", regex=True).str.strip()
    return s

def _norm_name_france(s: pd.Series) -> pd.Series:
    s = _base_normalize_series(s)
    return _apply_abbrevs_series(s, _LEGAL_ABBREVS)

def _norm_addr_france(s: pd.Series) -> pd.Series:
    return _base_normalize_series(s)

def _norm_name_default(s: pd.Series) -> pd.Series:
    """Fallback for any unseen country (§4.1 open-set requirement)."""
    s = _base_normalize_series(s)
    return _apply_abbrevs_series(s, _LEGAL_ABBREVS)

def _norm_addr_default(s: pd.Series) -> pd.Series:
    """Fallback for any unseen country."""
    s = _base_normalize_series(s)
    return _apply_abbrevs_series(s, _ADDR_ABBREVS)


# Dispatch dict: lowercase country key -> (name_fn, addr_fn)
COUNTRY_RULES: Dict[str, Tuple[Callable, Callable]] = {
    "us":      (_norm_name_us,      _norm_addr_us),
    "india":   (_norm_name_india,   _norm_addr_india),
    "france":  (_norm_name_france,  _norm_addr_france),
    "default": (_norm_name_default, _norm_addr_default),
}

print("§2.2 COUNTRY_RULES defined — US, India, France + default fallback for open-set countries")


In [ ]:
# §2.3 — Vectorized normalization of a full DataFrame
def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add normalized_name, normalized_address, normalized_country,
    name_tokens, address_tokens columns to df (in-place copy).
    Groups by country key and dispatches to the appropriate handlers.
    Every country value — including unseen ones — is handled by the
    'default' fallback; no record is dropped.
    """
    out = df.copy()
    out["normalized_country"] = out["country"].str.lower().str.strip()
    out["normalized_name"]    = pd.Series("", index=out.index, dtype=str)
    out["normalized_address"] = pd.Series("", index=out.index, dtype=str)

    # Collect all distinct country keys
    distinct_countries = out["normalized_country"].unique()

    for ck in distinct_countries:
        name_fn, addr_fn = COUNTRY_RULES.get(ck, COUNTRY_RULES["default"])
        mask = out["normalized_country"] == ck
        out.loc[mask, "normalized_name"]    = name_fn(out.loc[mask, "business_name"]).values
        out.loc[mask, "normalized_address"] = addr_fn(out.loc[mask, "business_address"]).values

    # Tokenize
    out["name_tokens"]    = out["normalized_name"].str.split().apply(
        lambda x: x if isinstance(x, list) else []
    )
    out["address_tokens"] = out["normalized_address"].str.split().apply(
        lambda x: x if isinstance(x, list) else []
    )
    return out


print("§2.3 normalize_df() defined — running on S1, S2, S3 ...")
df_s1_norm = normalize_df(df_s1)
print(f"  S1 normalized: {len(df_s1_norm):,}")
df_s2_norm = normalize_df(df_s2)
print(f"  S2 normalized: {len(df_s2_norm):,}")
df_s3_norm = normalize_df(df_s3)
print(f"  S3 normalized: {len(df_s3_norm):,}")

# §1→§2 handoff variable: normalized_records (combined, for TF-IDF vectorizer fitting)
# We keep them as DataFrames for vectorized access; NormalizedRecord objects are
# built on-the-fly only when needed (blocking key generation) to stay memory-efficient.
print(f"  All sources normalized.")


In [ ]:
# §2 SANITY CHECK
assert len(df_s1_norm) == len(df_s1)
assert len(df_s2_norm) == len(df_s2)
assert len(df_s3_norm) == len(df_s3)

# Open-set country: unseen value must go through fallback (not raise, not be dropped)
_dummy = pd.DataFrame({"entity_id": ["S1-X"], "business_name": ["Acme Corp"],
                        "business_address": ["12 Main St"], "country": ["Germany"],
                        "source": ["S1"]})
_dnorm = normalize_df(_dummy)
assert len(_dnorm) == 1, "Unseen country caused record to be dropped"
assert _dnorm["normalized_name"].iloc[0] != "", "Unseen country produced empty name"
assert "germany" not in COUNTRY_RULES, "'germany' must NOT be in COUNTRY_RULES (uses default)"

# Verify key columns present
for _col in ("normalized_name", "normalized_address", "normalized_country",
             "name_tokens", "address_tokens"):
    assert _col in df_s1_norm.columns, f"Missing column: {_col}"

print("§2 SANITY CHECK PASSED — including unseen-country fallback assertion")


---
## §3 Blocking

Generates blocking keys, builds an inverted index over S2/S3, and retrieves candidates per S1 entity. `candidate_pairs` (the final-stage candidate set) is used by both §4 (in-memory) and §7 (`candidate_pairs.tsv`).

In [ ]:
# §3.1 — Blocking key generators
_BLK          = CONFIG["blocking"]
_NAME_PFX_LEN = _BLK["name_prefix_len"]
_USE_SOUNDEX  = _BLK["use_soundex"]
_USE_ZIP      = _BLK["use_address_zip"]
_MAX_CAND     = _BLK["max_candidates_per_entity"]

_ZIP_RE = re.compile(r"\b(\d{5,6})\b")

def generate_blocking_keys(norm_name: str, norm_addr: str, norm_country: str) -> List[str]:
    """
    Generate blocking keys for one record.  Key types:
      cp:<country>:<sorted-top3-token-prefix>  — name fingerprint
      ct:<country>:<per-token-prefix>          — individual name tokens
      sdx:<country>:<soundex-of-1st-token>     — phonetic (if enabled)
      zip:<5-digit-zip-or-pin>                 — ZIP/PIN (if enabled)
    """
    keys = []
    country = norm_country or "unknown"
    ntoks   = norm_name.split() if norm_name else []

    # Key 1: sorted fingerprint of top-3 name tokens
    if ntoks:
        top3 = sorted(ntoks[:3])
        fp   = " ".join(t[:_NAME_PFX_LEN] for t in top3)
        keys.append(f"cp:{country}:{fp}")

    # Key 2: per-token prefix (up to 5 tokens, min length 3)
    for tok in ntoks[:5]:
        if len(tok) >= 3:
            keys.append(f"ct:{country}:{tok[:_NAME_PFX_LEN]}")

    # Key 3: Soundex of first name token
    if _USE_SOUNDEX and ntoks:
        try:
            sdx = jellyfish.soundex(ntoks[0])
            if sdx and sdx != "0000":
                keys.append(f"sdx:{country}:{sdx}")
        except Exception:
            pass

    # Key 4: ZIP / PIN code
    if _USE_ZIP and norm_addr:
        m = _ZIP_RE.search(norm_addr)
        if m:
            digits = m.group(1)[:5]
            if len(digits) >= 5:
                keys.append(f"zip:{digits}")

    return list(dict.fromkeys(keys))  # deduplicate, preserve order


print("§3.1 generate_blocking_keys() defined")


In [ ]:
# §3.2 — Assign blocking keys via vectorized apply
print("Generating blocking keys for S1, S2, S3 ...")

def _bkeys_for_df(df: pd.DataFrame) -> pd.Series:
    return pd.Series(
        [
            generate_blocking_keys(row.normalized_name, row.normalized_address, row.normalized_country)
            for row in df[["normalized_name", "normalized_address", "normalized_country"]].itertuples(index=False)
        ],
        index=df.index,
    )

df_s1_norm["blocking_keys"] = _bkeys_for_df(df_s1_norm)
df_s2_norm["blocking_keys"] = _bkeys_for_df(df_s2_norm)
df_s3_norm["blocking_keys"] = _bkeys_for_df(df_s3_norm)
print(f"  Blocking keys generated. Sample S1: {df_s1_norm['blocking_keys'].iloc[0][:3]}")


In [ ]:
# §3.3 — Build inverted index over S2 + S3
# Check cache first
_CACHE_ENABLED = _BLK.get("cache_enabled", False)
_CACHE_PATH    = (_NB_DIR / _BLK["cache_path"]).resolve() if "cache_path" in _BLK else None

if _CACHE_ENABLED and _CACHE_PATH and _CACHE_PATH.exists():
    print(f"Loading blocking cache from {_CACHE_PATH} ...")
    _cache_df   = pd.read_parquet(_CACHE_PATH)
    _cp_list_raw = _cache_df.values.tolist()
    candidate_pairs: List[CandidatePair] = [
        CandidatePair(
            source1_entity_id=r[0], candidate_entity_id=r[1],
            candidate_source=r[2],
            matched_blocking_keys=r[3].split("|") if r[3] else [],
        )
        for r in _cp_list_raw
    ]
    print(f"  Loaded {len(candidate_pairs):,} pairs from cache")
else:
    print("Building inverted index over S2 + S3 ...")
    inverted_index: Dict[str, List[Tuple[str, str]]] = collections.defaultdict(list)
    for df_src in (df_s2_norm, df_s3_norm):
        for eid, bkeys in zip(df_src["entity_id"], df_src["blocking_keys"]):
            src_prefix = eid.split("-")[0]
            for key in bkeys:
                inverted_index[key].append((eid, src_prefix))
    print(f"  Index size: {len(inverted_index):,} distinct keys")

    print("Running blocking lookup for all S1 entities ...")

    def build_candidate_pairs(
        s1_df: pd.DataFrame,
        idx: Dict[str, List[Tuple[str, str]]],
        max_cand: int,
    ) -> List[CandidatePair]:
        pairs = []
        for s1id, bkeys in zip(s1_df["entity_id"], s1_df["blocking_keys"]):
            cand_keys: Dict[str, List[str]] = collections.defaultdict(list)
            for bkey in bkeys:
                for (cid, _csrc) in idx.get(bkey, []):
                    cand_keys[cid].append(bkey)
            if len(cand_keys) > max_cand:
                sorted_c = sorted(cand_keys.items(), key=lambda x: (-len(x[1]), x[0]))
                sorted_c = sorted_c[:max_cand]
            else:
                sorted_c = list(cand_keys.items())
            for cid, mkeys in sorted_c:
                pairs.append(CandidatePair(
                    source1_entity_id=s1id,
                    candidate_entity_id=cid,
                    candidate_source=cid.split("-")[0],
                    matched_blocking_keys=mkeys,
                ))
        return pairs

    candidate_pairs: List[CandidatePair] = build_candidate_pairs(df_s1_norm, inverted_index, _MAX_CAND)
    print(f"  Generated {len(candidate_pairs):,} candidate pairs")

    if _CACHE_ENABLED and _CACHE_PATH:
        _cdf = pd.DataFrame([
            {"source1_entity_id": cp.source1_entity_id,
             "candidate_entity_id": cp.candidate_entity_id,
             "candidate_source": cp.candidate_source,
             "matched_blocking_keys": "|".join(cp.matched_blocking_keys)}
            for cp in candidate_pairs
        ])
        _cdf.to_parquet(_CACHE_PATH, index=False)
        print(f"  Cache saved to {_CACHE_PATH}")


In [ ]:
# §3.4 — Build candidate set index (s1_id -> set of candidate ids)
candidate_set_index: Dict[str, set] = collections.defaultdict(set)
for cp in candidate_pairs:
    candidate_set_index[cp.source1_entity_id].add(cp.candidate_entity_id)

# Ensure every S1 entity has an entry (even empty)
_s1_ids = set(df_s1_norm["entity_id"])
for sid in _s1_ids:
    if sid not in candidate_set_index:
        candidate_set_index[sid] = set()

_with_cands = sum(1 for v in candidate_set_index.values() if v)
print(f"  S1 entities with ≥1 candidate: {_with_cands:,} / {len(_s1_ids):,}")


In [ ]:
# §3 SANITY CHECK
_s23_ids = set(df_s2_norm["entity_id"]) | set(df_s3_norm["entity_id"])

_bad_s1  = sum(1 for cp in candidate_pairs if cp.source1_entity_id not in _s1_ids)
assert _bad_s1  == 0, f"{_bad_s1} pairs with unknown S1 id"

_bad_s23 = sum(1 for cp in candidate_pairs if cp.candidate_entity_id not in _s23_ids)
assert _bad_s23 == 0, f"{_bad_s23} pairs with unknown S2/S3 id"

_self    = sum(1 for cp in candidate_pairs if cp.candidate_entity_id.startswith("S1-"))
assert _self == 0, "Self-matches in candidate pairs"

assert len(candidate_set_index) == len(_s1_ids), (
    f"candidate_set_index covers {len(candidate_set_index)} S1s, expected {len(_s1_ids)}"
)
print(f"§3 SANITY CHECK PASSED — {len(candidate_pairs):,} pairs, all refs valid")


---
## §4 Feature Engineering

Computes the 7 similarity features for every `CandidatePair` (§4.3 of `implementation.md`).

In [ ]:
# §4.1 — Build fast lookup tables (entity_id -> row dict)
print("Building record lookup tables ...")

# Store only the fields we need for feature computation
_LOOKUP_COLS = ["entity_id", "normalized_name", "normalized_address",
                "normalized_country", "name_tokens", "address_tokens"]

rec_lookup: Dict[str, dict] = {}
for df_src in (df_s1_norm, df_s2_norm, df_s3_norm):
    for row in df_src[_LOOKUP_COLS].itertuples(index=False):
        rec_lookup[row.entity_id] = {
            "norm_name":    row.normalized_name,
            "norm_addr":    row.normalized_address,
            "norm_country": row.normalized_country,
            "name_toks":    row.name_tokens,
            "addr_toks":    row.address_tokens,
        }

print(f"§4.1 rec_lookup built: {len(rec_lookup):,} entries")


In [ ]:
# §4.2 — Fit TF-IDF vectorizers (sampled for scale)
# On 12M+ records, fitting TF-IDF on everything is memory-prohibitive.
# We sample up to 500K records per source for vectorizer fitting;
# vectors for all records are computed after fitting.
_TFIDF_SAMPLE = 500_000

def _sample_texts(df: pd.DataFrame, col: str, n: int) -> List[str]:
    s = df[col]
    if len(s) > n:
        s = s.sample(n=n, random_state=RANDOM_SEED)
    return s.tolist()

print("Fitting TF-IDF vectorizers (sampled) ...")
_fit_names = (
    _sample_texts(df_s1_norm, "normalized_name", _TFIDF_SAMPLE)
    + _sample_texts(df_s2_norm, "normalized_name", _TFIDF_SAMPLE)
    + _sample_texts(df_s3_norm, "normalized_name", _TFIDF_SAMPLE)
)
_fit_addrs = (
    _sample_texts(df_s1_norm, "normalized_address", _TFIDF_SAMPLE)
    + _sample_texts(df_s2_norm, "normalized_address", _TFIDF_SAMPLE)
    + _sample_texts(df_s3_norm, "normalized_address", _TFIDF_SAMPLE)
)

tfidf_name = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200_000)
tfidf_name.fit(_fit_names)
tfidf_addr = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200_000)
tfidf_addr.fit(_fit_addrs)
del _fit_names, _fit_addrs

print(f"  Name vocab: {len(tfidf_name.vocabulary_):,}  Addr vocab: {len(tfidf_addr.vocabulary_):,}")

# Build sparse TF-IDF for only the entities that appear in candidate_pairs
# (much smaller than all 12M records)
_cp_eids = set()
for cp in candidate_pairs:
    _cp_eids.add(cp.source1_entity_id)
    _cp_eids.add(cp.candidate_entity_id)
print(f"  Unique entity IDs in candidate pairs: {len(_cp_eids):,}")

_cp_ids_ordered  = list(_cp_eids)
_cp_norm_names   = [rec_lookup.get(eid, {}).get("norm_name",   "") for eid in _cp_ids_ordered]
_cp_norm_addrs   = [rec_lookup.get(eid, {}).get("norm_addr",   "") for eid in _cp_ids_ordered]

_name_mat = tfidf_name.transform(_cp_norm_names)
_addr_mat = tfidf_addr.transform(_cp_norm_addrs)
_id_to_row = {eid: i for i, eid in enumerate(_cp_ids_ordered)}

print(f"  Name TF-IDF matrix: {_name_mat.shape}")
print(f"  Addr TF-IDF matrix: {_addr_mat.shape}")


In [ ]:
# §4.3 — Feature functions
FEATURE_NAMES: List[str] = CONFIG["features"]

def _jaccard(a: List[str], b: List[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa and not sb: return 1.0
    if not sa or  not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

def _lev_ratio(a: str, b: str) -> float:
    if not a and not b: return 1.0
    if not a or  not b: return 0.0
    return Levenshtein.ratio(a, b)

def _cosine(id1: str, id2: str, mat, id2row: dict) -> float:
    r1, r2 = id2row.get(id1), id2row.get(id2)
    if r1 is None or r2 is None: return 0.0
    v1, v2 = mat[r1], mat[r2]
    dot = v1.dot(v2.T).toarray()[0, 0]
    n1  = math.sqrt(v1.dot(v1.T).toarray()[0, 0])
    n2  = math.sqrt(v2.dot(v2.T).toarray()[0, 0])
    if n1 == 0 or n2 == 0: return 0.0
    return float(dot / (n1 * n2))


def compute_features(s1id: str, cid: str) -> Dict[str, float]:
    """Compute the 7 similarity features for one (S1, candidate) pair."""
    r1 = rec_lookup[s1id]
    r2 = rec_lookup[cid]
    return {
        "name_jaccard":            _jaccard(r1["name_toks"], r2["name_toks"]),
        "name_levenshtein_ratio":  _lev_ratio(r1["norm_name"], r2["norm_name"]),
        "name_tfidf_cosine":       _cosine(s1id, cid, _name_mat, _id_to_row),
        "address_jaccard":         _jaccard(r1["addr_toks"], r2["addr_toks"]),
        "address_levenshtein_ratio": _lev_ratio(r1["norm_addr"], r2["norm_addr"]),
        "address_tfidf_cosine":    _cosine(s1id, cid, _addr_mat, _id_to_row),
        "country_match":           1.0 if r1["norm_country"] == r2["norm_country"] else 0.0,
    }

print(f"§4.3 compute_features() defined — features: {FEATURE_NAMES}")


In [ ]:
# §4.4 — Compute feature matrix for all candidate pairs
print(f"Computing {len(FEATURE_NAMES)} features for {len(candidate_pairs):,} candidate pairs ...")

# Build feature matrix directly (avoid creating ScoredPair objects yet for memory)
_n_pairs  = len(candidate_pairs)
_n_feats  = len(FEATURE_NAMES)
_feat_mat = np.zeros((_n_pairs, _n_feats), dtype=np.float32)

for i, cp in enumerate(candidate_pairs):
    feats = compute_features(cp.source1_entity_id, cp.candidate_entity_id)
    for j, fname in enumerate(FEATURE_NAMES):
        _feat_mat[i, j] = feats[fname]

print(f"  Feature matrix: {_feat_mat.shape}")

# Now create ScoredPair list with features embedded
scored_pairs: List[ScoredPair] = [
    ScoredPair(
        source1_entity_id   = cp.source1_entity_id,
        candidate_entity_id = cp.candidate_entity_id,
        candidate_source    = cp.candidate_source,
        matched_blocking_keys = cp.matched_blocking_keys,
        features = {fn: float(_feat_mat[i, j]) for j, fn in enumerate(FEATURE_NAMES)},
        match_probability = 0.0,
        predicted_label   = False,
    )
    for i, cp in enumerate(candidate_pairs)
]
print(f"  ScoredPairs created (pre-score): {len(scored_pairs):,}")


In [ ]:
# §4 SANITY CHECK
assert len(scored_pairs) == len(candidate_pairs)
if scored_pairs:
    _sp0 = scored_pairs[0]
    assert set(_sp0.features.keys()) == set(FEATURE_NAMES), "Feature key mismatch"
    for _fn in FEATURE_NAMES:
        assert 0.0 <= _sp0.features[_fn] <= 1.0, f"Feature {_fn} out of [0,1]"
print("§4 SANITY CHECK PASSED")


---
## §5 Matching Model

Trains a `GradientBoostingClassifier` on ground-truth positives + sampled negatives (train mode). Scores all candidate pairs in both modes.

In [ ]:
# §5.1 — Model configuration
_MC        = CONFIG["matching_model"]
_THRESHOLD = float(_MC["threshold"])
_NEG_RATIO = int(_MC["neg_sample_ratio"])
_GBC_PARAMS = dict(
    n_estimators  = _MC["n_estimators"],
    max_depth     = _MC["max_depth"],
    learning_rate = _MC["learning_rate"],
    subsample     = _MC["subsample"],
    random_state  = RANDOM_SEED,
)
_MODEL_PATH = MODELS_DIR / "classifier.pkl"

print(f"§5.1 Model : {_MC['name']}")
print(f"     License: {_MC['license']}  |  Params: {_MC['parameter_count']}")
print(f"     Threshold: {_THRESHOLD}  |  Neg-ratio: {_NEG_RATIO}")


In [ ]:
# §5.2 — Train or load classifier, then score
def train_matcher(
    feat_matrix: np.ndarray,
    pairs: List[ScoredPair],
    lbl_dict: Dict[Tuple[str, str], int],
    gbc_params: dict,
    neg_ratio: int,
    seed: int,
    s1_train_ids: Optional[set] = None,
) -> GradientBoostingClassifier:
    # Filter to train split
    if s1_train_ids:
        idx_keep = [i for i, p in enumerate(pairs) if p.source1_entity_id in s1_train_ids]
    else:
        idx_keep = list(range(len(pairs)))

    pos_idx = [i for i in idx_keep if lbl_dict.get((pairs[i].source1_entity_id, pairs[i].candidate_entity_id), 0) == 1]
    neg_pool = [i for i in idx_keep if lbl_dict.get((pairs[i].source1_entity_id, pairs[i].candidate_entity_id), 0) == 0]

    n_neg = min(len(neg_pool), len(pos_idx) * neg_ratio)
    rng   = random.Random(seed)
    neg_idx = rng.sample(neg_pool, n_neg)

    train_idx = pos_idx + neg_idx
    y = np.array([1] * len(pos_idx) + [0] * len(neg_idx))
    X = feat_matrix[train_idx]

    print(f"  Training: {len(pos_idx)} pos + {len(neg_idx)} neg = {len(y)} samples")
    clf = GradientBoostingClassifier(**gbc_params)
    clf.fit(X, y)
    return clf


def score_pairs_inplace(
    pairs: List[ScoredPair],
    feat_matrix: np.ndarray,
    clf: GradientBoostingClassifier,
    threshold: float,
) -> None:
    """Score all ScoredPairs in-place using the pre-computed feature matrix."""
    if not pairs:
        return
    probs = clf.predict_proba(feat_matrix)[:, 1]
    for i, sp in enumerate(pairs):
        sp.match_probability = float(probs[i])
        sp.predicted_label   = bool(probs[i] >= threshold)


print("§5.2 train_matcher(), score_pairs_inplace() defined")


In [ ]:
# §5.3 — Execute: train (train mode) or load saved / use heuristic (test mode)
if RUN_MODE == "train" and labels:
    print("Train mode: stratified train/val split of S1 entities ...")
    _s1_ids_list = df_s1_norm["entity_id"].tolist()
    _singleton_flags = [0 if (gt_map or {}).get(sid) else 1 for sid in _s1_ids_list]

    _s1_train_list, _s1_val_list, _, _ = train_test_split(
        _s1_ids_list, _singleton_flags,
        test_size    = CONFIG["val_split_ratio"],
        stratify     = _singleton_flags,
        random_state = CONFIG["val_split_seed"],
    )
    _s1_train_ids = set(_s1_train_list)
    _s1_val_ids   = set(_s1_val_list)
    print(f"  Train S1: {len(_s1_train_ids):,}  Val S1: {len(_s1_val_ids):,}")

    print("  Fitting GradientBoostingClassifier ...")
    classifier = train_matcher(
        _feat_mat, scored_pairs, labels, _GBC_PARAMS, _NEG_RATIO, RANDOM_SEED,
        s1_train_ids=_s1_train_ids,
    )
    print("  Scoring all candidate pairs ...")
    score_pairs_inplace(scored_pairs, _feat_mat, classifier, _THRESHOLD)

    # Save classifier for test-mode re-use
    with open(_MODEL_PATH, "wb") as _mf:
        pickle.dump(classifier, _mf)
    print(f"  Classifier saved: {_MODEL_PATH}")

elif _MODEL_PATH.exists():
    print(f"Test mode: loading saved classifier from {_MODEL_PATH} ...")
    with open(_MODEL_PATH, "rb") as _mf:
        classifier = pickle.load(_mf)
    score_pairs_inplace(scored_pairs, _feat_mat, classifier, _THRESHOLD)

else:
    print("Test mode: no saved classifier — using weighted feature heuristic ...")
    classifier = None
    # Weights: name features dominate (most discriminative)
    _W = np.array([0.35, 0.25, 0.20, 0.10, 0.05, 0.05, 0.00], dtype=np.float32)
    _scores = _feat_mat @ _W
    for i, sp in enumerate(scored_pairs):
        sp.match_probability = float(_scores[i])
        sp.predicted_label   = bool(_scores[i] >= _THRESHOLD)

_n_pos = sum(1 for sp in scored_pairs if sp.predicted_label)
print(f"  {_n_pos:,} pairs predicted positive ({100*_n_pos/max(1,len(scored_pairs)):.1f}%)")


In [ ]:
# §5 SANITY CHECK
assert len(scored_pairs) == len(candidate_pairs)
for _sp in scored_pairs[:200]:
    assert 0.0 <= _sp.match_probability <= 1.0
    assert isinstance(_sp.predicted_label, (bool, np.bool_))
print("§5 SANITY CHECK PASSED")


---
## §6 Aggregation

Thresholds, groups by `source1_entity_id`, deduplicates, enforces `matched ⊆ candidates` invariant, and emits exactly one `FinalMatch` per S1 entity.

In [ ]:
# §6.1 — Aggregation
def aggregate_matches(
    pairs: List[ScoredPair],
    s1_ids_ordered: List[str],
    cand_index: Dict[str, set],
) -> List[FinalMatch]:
    """
    Groups positive predictions by S1 entity, deduplicates, enforces
    matched ⊆ candidates, emits one FinalMatch per S1 entity.
    """
    predicted: Dict[str, List[str]] = collections.defaultdict(list)
    for sp in pairs:
        if sp.predicted_label:
            predicted[sp.source1_entity_id].append(sp.candidate_entity_id)

    out = []
    for sid in s1_ids_ordered:
        raw = predicted.get(sid, [])
        # Deduplicate
        seen: set = set()
        dedup = []
        for mid in raw:
            if mid not in seen:
                seen.add(mid)
                dedup.append(mid)
        # Enforce matched ⊆ candidates (invariant §2.4)
        cands   = cand_index.get(sid, set())
        valid   = [mid for mid in dedup if mid in cands]
        out.append(FinalMatch(source1_entity_id=sid, matched_entity_ids=valid))
    return out


print("§6.1 aggregate_matches() defined — aggregating ...")
_s1_ids_ordered = df_s1_norm["entity_id"].tolist()
final_matches: List[FinalMatch] = aggregate_matches(scored_pairs, _s1_ids_ordered, candidate_set_index)

_n_empty = sum(1 for fm in final_matches if not fm.matched_entity_ids)
print(f"  FinalMatch records : {len(final_matches):,}")
print(f"  Singletons (empty) : {_n_empty:,}")
print(f"  With matches       : {len(final_matches)-_n_empty:,}")


In [ ]:
# §6 SANITY CHECK
assert len(final_matches) == len(_s1_ids_ordered), "FinalMatch count != S1 count"

_fm_ids = [fm.source1_entity_id for fm in final_matches]
assert len(_fm_ids) == len(set(_fm_ids)), "Duplicate S1 IDs in final_matches"
assert set(_fm_ids) == _s1_ids, "FinalMatch S1 ID set mismatch"

for fm in final_matches:
    assert len(fm.matched_entity_ids) == len(set(fm.matched_entity_ids)), \
        f"Duplicate matched IDs for {fm.source1_entity_id}"
    for mid in fm.matched_entity_ids:
        assert not mid.startswith("S1-"), f"Self-match: {mid}"
        assert mid.startswith(("S2-", "S3-")), f"Bad prefix: {mid}"
        # matched ⊆ candidates invariant
        assert mid in candidate_set_index[fm.source1_entity_id], \
            f"Invariant violated: {mid} not in candidates for {fm.source1_entity_id}"

print("§6 SANITY CHECK PASSED")


---
## §7 Output Writing

Serializes both output files to `output/` in the exact format required by §4.4.

In [ ]:
# §7.1 — Output writers
def write_matching_results(matches: List[FinalMatch], out_path: pathlib.Path) -> None:
    """Tab-separated; header row; empty string for singletons; no quoting."""
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        f.write("source1_entity_id\tmatched_entity_ids\n")
        for fm in matches:
            f.write(f"{fm.source1_entity_id}\t{','.join(fm.matched_entity_ids)}\n")


def write_candidate_pairs(
    s1_ids: List[str],
    cand_idx: Dict[str, set],
    out_path: pathlib.Path,
) -> None:
    """Tab-separated; header row; empty string for no-candidate entities; no quoting."""
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        f.write("source1_entity_id\tcandidate_entity_ids\n")
        for sid in s1_ids:
            cands = sorted(cand_idx.get(sid, set()))
            f.write(f"{sid}\t{','.join(cands)}\n")


print("§7.1 writers defined")


In [ ]:
# §7.2 — Write output files
_matching_path  = OUTPUT_DIR / "matching_results.tsv"
_candidate_path = OUTPUT_DIR / "candidate_pairs.tsv"

print(f"Writing matching_results.tsv  -> {_matching_path}")
write_matching_results(final_matches, _matching_path)
print(f"  Done: {len(final_matches):,} rows")

print(f"Writing candidate_pairs.tsv   -> {_candidate_path}")
write_candidate_pairs(_s1_ids_ordered, candidate_set_index, _candidate_path)
print(f"  Done: {len(_s1_ids_ordered):,} rows")


In [ ]:
# §7 SANITY CHECK
assert _matching_path.exists()
assert _candidate_path.exists()

with open(_matching_path, "r", encoding="utf-8") as _f:
    _mlines = _f.readlines()
assert _mlines[0].rstrip("\n") == "source1_entity_id\tmatched_entity_ids", \
    f"Bad matching header: {_mlines[0]!r}"
assert len(_mlines) == len(_s1_ids_ordered) + 1, \
    f"matching_results.tsv has {len(_mlines)-1} rows, expected {len(_s1_ids_ordered)}"
for _l in _mlines[1:5]:
    assert "\t" in _l, f"Missing TAB in: {_l!r}"

with open(_candidate_path, "r", encoding="utf-8") as _f:
    _clines = _f.readlines()
assert _clines[0].rstrip("\n") == "source1_entity_id\tcandidate_entity_ids"
assert len(_clines) == len(_s1_ids_ordered) + 1

print("§7 SANITY CHECK PASSED")


---
## §8 Validation

Shells out to `utils/validate_submission.py` and asserts exit code 0.

In [ ]:
# §8.1 — Run the organizer's validator
def run_validator() -> int:
    _student_resource_dir = OUTPUT_DIR.parent
    _validator = _student_resource_dir / "utils" / "validate_submission.py"
    # In train mode validate against train source1; in test mode against test source1
    _test_dir  = _student_resource_dir / "dataset" / RUN_MODE

    if not _validator.exists():
        print(f"WARNING: Validator not found at {_validator} — skipping")
        return 0

    cmd = [
        sys.executable, str(_validator),
        "--matching",   str(_matching_path),
        "--candidate",  str(_candidate_path),
        "--test-dir",   str(_test_dir),
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print("STDERR:", res.stderr[:500])
    return res.returncode


_validator_exit_code = run_validator()
print(f"Validator exit code: {_validator_exit_code}")


In [ ]:
# §8 SANITY CHECK
assert _validator_exit_code == 0, (
    f"Validator exit code {_validator_exit_code} — fix output files before packaging."
)
print("§8 SANITY CHECK PASSED — validator returned 0 (PASS)")


---
## §9 Evaluation (dev only)

Macro-averaged F₀.₅ on held-out validation split — **only executes in train mode when ground truth is present**.

In [ ]:
# §9.1 — F0.5 metric per §2.6 of implementation.md
def entity_f05(true_ids: set, pred_ids: set) -> float:
    """Compute per-entity F0.5 using the exact rules from §2.6."""
    if not true_ids and not pred_ids: return 1.0
    if not true_ids and pred_ids:     return 0.0
    if not pred_ids:
        prec, rec = 0.0, 0.0
    else:
        tp   = len(true_ids & pred_ids)
        prec = tp / len(pred_ids)
        rec  = tp / len(true_ids)
    if prec == 0 and rec == 0:
        return 0.0
    return (1.25 * prec * rec) / (0.25 * prec + rec)


def evaluate_holdout(
    fm_list: List[FinalMatch],
    gt: Dict[str, List[str]],
    val_ids: set,
) -> float:
    """Macro-averaged F0.5 over the validation S1 entities."""
    fm_map = {fm.source1_entity_id: set(fm.matched_entity_ids) for fm in fm_list}
    scores = [
        entity_f05(set(gt.get(sid, [])), fm_map.get(sid, set()))
        for sid in val_ids
    ]
    return float(np.mean(scores)) if scores else 0.0


print("§9.1 F0.5 functions defined")


In [ ]:
# §9.2 — Run evaluation (guarded: train mode + gt only)
if RUN_MODE == "train" and gt_map is not None:
    print("=" * 60)
    print("§9 EVALUATION — held-out macro-averaged F0.5")
    print("=" * 60)
    macro_f05 = evaluate_holdout(final_matches, gt_map, _s1_val_ids)
    print(f"\n  Validation S1 entities : {len(_s1_val_ids):,}")
    print(f"  Macro-averaged F0.5    : {macro_f05:.6f}")
    print("=" * 60)
else:
    macro_f05 = None
    print("§9 EVALUATION: Skipped — test mode (no ground truth)")


In [ ]:
# §9 SANITY CHECK
if RUN_MODE == "train" and macro_f05 is not None:
    assert 0.0 <= macro_f05 <= 1.0, f"macro_f05 {macro_f05} out of [0,1]"
    print(f"§9 SANITY CHECK PASSED — macro F0.5 = {macro_f05:.4f}")
else:
    print("§9 SANITY CHECK PASSED (skipped in test mode)")

print()
print("=" * 60)
print("Pipeline complete.")
print(f"  matching_results.tsv : {_matching_path}")
print(f"  candidate_pairs.tsv  : {_candidate_path}")
print("=" * 60)
